# 04 - Extração estruturada via LLM

Para cada PDF baixado no notebook 03, extrai full text com `pymupdf` e roda extração estruturada via DeepSeek produzindo as linhas da tabela `configurations` (schema em `00_design.ipynb`).

**Saída:**
- `data/processed/extraction_checkpoints.jsonl` - uma linha por paper com a resposta crua do LLM.
- `data/processed/configurations_raw.parquet` - base achatada: uma linha por configuração experimental extraída (1 paper → N linhas), com todos os campos `*_raw`.

## Decisões

- **Parser de PDF:** `pymupdf` (rápido, simples). Se descobrirmos que tabelas de resultados estão sendo perdidas, escalamos para GROBID nos casos problemáticos.
- **Truncagem inteligente:** muitos papers passam de 50K chars. Para conter custo, truncamos para ~30K chars total = primeiros 10K (intro + métodos) + últimos 20K (experimentos + resultados + conclusão). Também tentamos cortar a seção de Referências quando detectada.
- **Modelo LLM:** `deepseek-v4-pro` — extração estruturada com muitos campos por linha; verifique o preço atual em platform.deepseek.com.
- **Structured output:** `instructor` + `client.chat.completions.create_with_completion()` com schema Pydantic aninhado (`PaperExtraction` contendo `list[Configuration]`).
- **Rate limit:** mesmo limiter do notebook 02 (45 RPM, margem abaixo dos 50).

In [ ]:
!pip install --upgrade pymupdf pymupdf4llm openai json-repair

In [3]:
import os
import json
import time
import threading
from collections import deque
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, List

import pandas as pd
import pymupdf4llm
import json_repair
from tqdm.auto import tqdm
from pydantic import BaseModel, Field, ValidationError
from openai import OpenAI

assert os.environ.get("DEEPSEEK_API_KEY")
client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)

c:\Users\fredb\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
NB_DIR = Path.cwd()
PROJECT_DIR = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PDF_DIR = PROJECT_DIR / "pdfs"

MODEL = "deepseek-v4-pro"
N_WORKERS = 4
RPM_LIMIT = 50
MAX_TOKENS_OUTPUT = 8000
TEMPERATURE = 0.0

MAX_TEXT_CHARS = 30000
HEAD_CHARS = 10000

PRICE_INPUT_PER_M = 0.435
PRICE_OUTPUT_PER_M = 0.87

INPUT_PATH = PROCESSED_DIR / "papers_with_pdf.parquet"
CKPT_PATH = PROCESSED_DIR / "extraction_checkpoints.jsonl"
CONFIG_PATH = PROCESSED_DIR / "configurations_raw.parquet"

print(f"Modelo: {MODEL}")
print(f"Rate limit: {RPM_LIMIT} req/min, {N_WORKERS} workers")
print(f"Truncagem: head={HEAD_CHARS} + tail={MAX_TEXT_CHARS - HEAD_CHARS} = {MAX_TEXT_CHARS} chars")
print(f"\nInput:  {INPUT_PATH}")
print(f"Ckpt:   {CKPT_PATH}")
print(f"Saída:  {CONFIG_PATH}")

Modelo: deepseek-v4-pro
Rate limit: 50 req/min, 4 workers
Truncagem: head=10000 + tail=20000 = 30000 chars

Input:  c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\causal_project\data\processed\papers_with_pdf.parquet
Ckpt:   c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\causal_project\data\processed\extraction_checkpoints.jsonl
Saída:  c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\causal_project\data\processed\configurations_raw.parquet


In [5]:
class RateLimiter:
    def __init__(self, max_per_minute: int):
        self.max_per_minute = max_per_minute
        self.timestamps: deque = deque()
        self.lock = threading.Lock()

    def acquire(self):
        while True:
            with self.lock:
                now = time.monotonic()
                while self.timestamps and self.timestamps[0] <= now - 60:
                    self.timestamps.popleft()
                if len(self.timestamps) < self.max_per_minute:
                    self.timestamps.append(now)
                    return
                sleep_time = self.timestamps[0] + 60 - now + 0.05
            time.sleep(max(sleep_time, 0.01))


rate_limiter = RateLimiter(RPM_LIMIT)

In [6]:
df = pd.read_parquet(INPUT_PATH)
print(f"Total: {len(df)}")
print(f"\nStatus do PDF:")
print(df["pdf_status"].value_counts(dropna=False))

df_ext = df[df["pdf_status"].isin(["success", "already_exists"])].copy().reset_index(drop=True)
print(f"\nA extrair: {len(df_ext)}")

Total: 1405

Status do PDF:
pdf_status
success          602
failed           470
not_attempted    327
no_url             6
Name: count, dtype: int64

A extrair: 602


## Extração de texto do PDF

Estratégia:

1. Extrai texto de todas as páginas.
2. Procura cabeçalho de Referências (`\nReferences`, `\nREFERENCES`, `\nBibliography`) na segunda metade do texto - se encontrar, descarta da posição em diante.
3. Se o texto restante for ≤ MAX_TEXT_CHARS, devolve inteiro.
4. Caso contrário, devolve `[head] + "[...middle truncated...]" + [tail]` - mantém início (abstract, intro, métodos) e fim (resultados, discussão, conclusão).

In [7]:
REFERENCES_PATTERNS = [
    "\nReferences\n", "\nREFERENCES\n", "\nReferences \n",
    "\nBibliography\n", "\nBIBLIOGRAPHY\n", "\nReferences:\n",
    "\n## **References**", "\n## References", "\n# References",
    "\n## **Bibliography**", "\n## Bibliography",
]


def extract_pdf_text(pdf_path: str) -> str:
    return pymupdf4llm.to_markdown(str(pdf_path))


def smart_truncate(text: str, max_chars: int = MAX_TEXT_CHARS,
                    head_chars: int = HEAD_CHARS) -> str:
    half = len(text) // 2
    for pat in REFERENCES_PATTERNS:
        idx = text.rfind(pat)
        if idx > half:
            text = text[:idx]
            break

    if len(text) <= max_chars:
        return text
    tail_chars = max_chars - head_chars
    return (text[:head_chars]
            + "\n\n[...middle truncated for length...]\n\n"
            + text[-tail_chars:])


if len(df_ext) > 0:
    sample_path = df_ext.iloc[3]["pdf_local_path"]
    print(f"Teste com: {sample_path}")
    raw = extract_pdf_text(sample_path)
    truncated = smart_truncate(raw)
    print(f"  raw:       {len(raw):>7} chars (~{len(raw)//4:>5} tokens)")
    print(f"  truncated: {len(truncated):>7} chars (~{len(truncated)//4:>5} tokens)")
    print(f"\nPrimeiros 500 chars:\n{truncated[:500]}")

Teste com: c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\causal_project\pdfs\arxiv_2002.04592.pdf
  raw:         73431 chars (~18357 tokens)
  truncated:   30039 chars (~ 7509 tokens)

Primeiros 500 chars:
# Imbalanced a classification: paradigm-based review 

# Yang Feng[1] , Min Zhou[2] , Xin Tong[3][∗] 

> 1 New York University 

> 2 BNU-HKBU United International College 

> 3 University of Southern California 

## Abstract 

A common issue for classification in scientific research and industry is the existence of imbalanced classes. When sample sizes of different classes are imbalanced in training data, naively implementing a classification method often leads to unsatisfactory prediction resul


## Schema da extração

Cada paper pode produzir múltiplas `Configuration`. O LLM devolve um `PaperExtraction` com lista de configs + um `paper_notes` opcional (útil quando o paper acaba não tendo dado experimental utilizável).

Todos os campos `*_raw` são strings literais do paper — normalização é separada (notebook 05). Os 3 campos meta (`extracted_evidence`, `extraction_confidence`, `notes`) são auxiliares pra auditoria.

In [8]:
class Configuration(BaseModel):
    dataset_name_raw: str = Field(description="Dataset name verbatim from the paper.")
    model_name_raw: str = Field(description="Model architecture name verbatim from the paper.")
    balancing_strategy_raw: str = Field(description="Class-balancing strategy verbatim (or 'baseline' if no balancing).")
    metric_name_raw: str = Field(description="Metric name verbatim (e.g., 'macro F1', 'Balanced Accuracy').")
    metric_value: float = Field(description="Numerical metric value. Convert percentages to decimals (e.g., 84.5% -> 0.845).")
    is_baseline_raw: bool = Field(description="True if paper explicitly marks this as a no-balancing baseline.")

    dataset_size_raw: Optional[str] = Field(default=None, description="Number of instances as written (e.g., '70,000 samples'). Null if not stated.")
    dataset_num_classes_raw: Optional[str] = Field(default=None, description="Number of classes as written. Null if not stated.")
    dataset_imbalance_ratio_raw: Optional[str] = Field(default=None, description="Imbalance ratio or class distribution as written (e.g., 'IR=100', 'minority 5%'). Null if not stated.")
    dataset_domain_raw: Optional[str] = Field(default=None, description="Domain (e.g., 'medical imaging', 'fraud detection'). Null if not clear.")
    task_type_raw: Optional[str] = Field(default=None, description="Task type (e.g., 'binary classification'). Null if not specified.")

    model_hparams_raw: Optional[str] = Field(default=None, description="Hyperparameters/training setup if mentioned. Null otherwise.")

    metric_split_raw: Optional[str] = Field(default=None, description="'test', 'val', 'cv' or other split as written. Null if unclear.")
    metric_aggregation_raw: Optional[str] = Field(default=None, description="How the value was aggregated (e.g., 'mean', 'mean±std', 'best'). Null if unclear.")

    extracted_evidence: str = Field(description="Short verbatim quote (1-2 sentences) from the paper supporting this extraction.")
    extraction_confidence: float = Field(description="Your confidence 0.0-1.0 that this is a correct, complete extraction.")
    notes: Optional[str] = Field(default=None, description="Anything notable about this configuration. Null otherwise.")


class PaperExtraction(BaseModel):
    configurations: List[Configuration] = Field(description="All experimental configurations extracted from the paper. Empty list if paper has no usable data.")
    paper_notes: Optional[str] = Field(default=None, description="General notes about the paper, especially if the configurations list is empty.")

In [9]:
SYSTEM_PROMPT = """You are extracting structured experimental data from machine learning papers that study class-balancing strategies for supervised classification. Respond with a single JSON object — no prose, no markdown fences.

For each paper, extract every experimental CONFIGURATION reported: a unique combination of (dataset, model, balancing strategy) measured by at least one metric value.

A single paper typically has multiple configurations. If a paper reports a results table like:
  Dataset A: ResNet + SMOTE=0.82, ResNet + Focal Loss=0.85, ResNet baseline=0.71
  Dataset B: ResNet + SMOTE=0.74, ResNet + Focal Loss=0.78, ResNet baseline=0.65
Then there are 6 configurations to extract.

REQUIRED for every configuration:
- dataset_name_raw: dataset name verbatim
- model_name_raw: model architecture verbatim
- balancing_strategy_raw: strategy name verbatim (or "baseline" / "no balancing" if that's the configuration)
- metric_name_raw: metric name verbatim (e.g., "macro F1", "Balanced Accuracy", "TPR gap", "AUPRC")
- metric_value: numerical value as a float. Convert percentages to decimals: "84.5%" -> 0.845
- is_baseline_raw: true only if the paper EXPLICITLY marks this as a no-balancing baseline
- extracted_evidence: short verbatim quote (1-2 sentences) supporting this extraction
- extraction_confidence: float 0.0-1.0

OPTIONAL (use null if not clearly stated):
- dataset_size_raw, dataset_num_classes_raw, dataset_imbalance_ratio_raw, dataset_domain_raw, task_type_raw
- model_hparams_raw, metric_split_raw, metric_aggregation_raw, notes

RULES:
1. SKIP configurations missing dataset, model, strategy, or metric value. Don't make up values.
2. Multiple metrics for the same (dataset, model, strategy) = MULTIPLE configurations (one per metric).
3. Extract EVERYTHING in results tables - do NOT filter to "important" ones.
4. If no usable experimental data exists, return configurations=[] and explain in paper_notes.
5. The paper text may be truncated in the middle - work with what's available.

OUTPUT — JSON with this exact shape:
{
  "configurations": [
    {
      "dataset_name_raw": "CIFAR-10-LT",
      "model_name_raw": "ResNet-32",
      "balancing_strategy_raw": "SMOTE",
      "metric_name_raw": "top-1 accuracy",
      "metric_value": 0.823,
      "is_baseline_raw": false,
      "dataset_size_raw": null,
      "dataset_num_classes_raw": "10",
      "dataset_imbalance_ratio_raw": "IR=100",
      "dataset_domain_raw": null,
      "task_type_raw": null,
      "model_hparams_raw": null,
      "metric_split_raw": "test",
      "metric_aggregation_raw": null,
      "extracted_evidence": "Table 2 shows ResNet-32 with SMOTE achieves 82.3% on CIFAR-10-LT.",
      "extraction_confidence": 0.95,
      "notes": null
    }
  ],
  "paper_notes": null
}"""

In [10]:
def _parse_json_lenient(content: str) -> dict:
    """Try strict json.loads first; fall back to json_repair if truncated/malformed."""
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        repaired = json_repair.loads(content)
        if not isinstance(repaired, dict):
            raise ValueError(f"json_repair returned non-dict: {type(repaired).__name__}")
        return repaired


def extract_configurations_from_paper(paper_id: str, pdf_path: str,
                                       model: str = MODEL) -> dict:
    base = {
        "paper_id": paper_id,
        "configurations": None,
        "paper_notes": None,
        "input_tokens": None,
        "output_tokens": None,
        "stop_reason": None,
        "n_configs": 0,
        "n_skipped": 0,
        "json_repaired": False,
        "error": None,
    }

    try:
        raw_text = extract_pdf_text(pdf_path)
        text = smart_truncate(raw_text)
    except Exception as e:
        base["error"] = f"pdf_extract_failed: {type(e).__name__}: {str(e)[:200]}"
        return base

    if len(text) < 500:
        base["error"] = f"text_too_short ({len(text)} chars after extraction)"
        return base

    user_msg = f"Paper text follows. Extract all experimental configurations as JSON.\n\n---\n\n{text}"
    rate_limiter.acquire()
    try:
        response = client.chat.completions.create(
            model=model,
            max_tokens=MAX_TOKENS_OUTPUT,
            temperature=TEMPERATURE,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg},
            ],
            response_format={"type": "json_object"},
            extra_body={"thinking": {"type": "disabled"}},
        )
        base["input_tokens"] = response.usage.prompt_tokens
        base["output_tokens"] = response.usage.completion_tokens
        base["stop_reason"] = response.choices[0].finish_reason

        content = response.choices[0].message.content
        try:
            data = json.loads(content)
        except json.JSONDecodeError:
            data = json_repair.loads(content)
            if not isinstance(data, dict):
                base["error"] = f"json_repair_non_dict: {type(data).__name__}"
                return base
            base["json_repaired"] = True

        raw_configs = data.get("configurations") or []
        good_configs = []
        for i, cfg in enumerate(raw_configs):
            try:
                good_configs.append(Configuration.model_validate(cfg).model_dump())
            except ValidationError:
                base["n_skipped"] += 1

        base["configurations"] = good_configs
        base["paper_notes"] = data.get("paper_notes")
        base["n_configs"] = len(good_configs)
        return base
    except Exception as e:
        base["error"] = f"{type(e).__name__}: {str(e)[:300]}"
        return base

## Smoke test - 3 papers

In [104]:
sample = df_ext.sample(n=3, random_state=42).reset_index(drop=True)

for row in sample.itertuples():
    print("=" * 80)
    print(f"[{row.rank}] {row.title[:90]}")
    print(f"  PDF: {row.pdf_local_path}")
    result = extract_configurations_from_paper(row.paper_id, row.pdf_local_path)
    if result["error"]:
        print(f"  ERRO: {result['error']}")
        continue
    flags = []
    if result.get("json_repaired"):
        flags.append("json_repaired")
    if result.get("stop_reason") == "length":
        flags.append("output_truncated")
    if result.get("n_skipped"):
        flags.append(f"skipped={result['n_skipped']}")
    flag_str = f"  [{', '.join(flags)}]" if flags else ""
    print(f"  configs extraídas: {result['n_configs']}{flag_str}")
    print(f"  tokens: in={result['input_tokens']} out={result['output_tokens']}  stop={result['stop_reason']}")
    if result["paper_notes"]:
        print(f"  paper_notes: {result['paper_notes'][:200]}")
    if result["n_configs"] > 0:
        print(f"  --- Primeira config: ---")
        first = result["configurations"][0]
        for k in ["dataset_name_raw", "model_name_raw", "balancing_strategy_raw",
                  "metric_name_raw", "metric_value", "is_baseline_raw",
                  "extracted_evidence", "extraction_confidence"]:
            print(f"    {k}: {first.get(k)}")
    print()

[605] Severely imbalanced Big Data challenges: investigating data sampling approaches
  PDF: c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\causal_project\pdfs\doi_10.1186_s40537-019-0274-4.pdf
  configs extraídas: 43  [json_repaired, output_truncated, skipped=1]
  tokens: in=11076 out=8000  stop=length
  --- Primeira config: ---
    dataset_name_raw: Medicare
    model_name_raw: GBT
    balancing_strategy_raw: RUS
    metric_name_raw: AUC
    metric_value: 0.79833
    is_baseline_raw: False
    extracted_evidence: Table 6 (a) AUC: GBT, RUS, AUC=0.79833
    extraction_confidence: 0.95

[2141] Heartbeat Anomaly Detection using Adversarial Oversampling
  PDF: c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\causal_project\pdfs\arxiv_1901.09972.pdf
  configs extraídas: 37  [json_repaired, output_truncated, skipped=1]
  tokens: in=8141 out=8000  stop=length
  --- Primeira config: ---
    dataset_name_raw: MIT-BIH Arrhythmia Database
    model_name_raw: CNN
    balancing_strategy_raw: O

## Run completo com checkpoint

In [11]:
def load_done_ids(ckpt_path: Path) -> set:
    if not ckpt_path.exists():
        return set()
    done = set()
    with open(ckpt_path, encoding="utf-8") as f:
        for line in f:
            try:
                entry = json.loads(line)
                if entry.get("error") is None and entry.get("configurations") is not None:
                    done.add(entry["paper_id"])
            except Exception:
                pass
    return done


def run_extraction(df: pd.DataFrame, ckpt_path: Path,
                    n_workers: int = N_WORKERS, model: str = MODEL):
    done_ids = load_done_ids(ckpt_path)
    todo = df[~df["paper_id"].isin(done_ids)].copy()
    print(f"Total: {len(df):>5}")
    print(f"Sucessos prévios: {len(done_ids):>5}")
    print(f"Restantes (inclui erros prévios): {len(todo):>5}")

    if len(todo) == 0:
        return 0

    est_tokens_in = len(todo) * 7500
    est_tokens_out = len(todo) * 1000
    est_cost = est_tokens_in / 1_000_000 * PRICE_INPUT_PER_M + est_tokens_out / 1_000_000 * PRICE_OUTPUT_PER_M
    est_time = len(todo) * 60 / RPM_LIMIT
    print(f"\nCusto estimado dos restantes: ~${est_cost:.2f}")
    print(f"Tempo mínimo (limitado por {RPM_LIMIT} RPM): ~{est_time/60:.1f} min\n")

    start = time.time()
    n_success = 0
    n_error = 0
    total_configs = 0

    with open(ckpt_path, "a", encoding="utf-8") as f_out:
        with ThreadPoolExecutor(max_workers=n_workers) as pool:
            futures = {
                pool.submit(extract_configurations_from_paper,
                            row.paper_id, row.pdf_local_path, model): row.paper_id
                for row in todo.itertuples()
            }
            for fut in tqdm(as_completed(futures), total=len(futures), desc="Extraindo"):
                try:
                    result = fut.result()
                except Exception as e:
                    result = {
                        "paper_id": futures[fut],
                        "configurations": None, "paper_notes": None,
                        "input_tokens": None, "output_tokens": None,
                        "stop_reason": None, "n_configs": 0,
                        "error": f"future_exception: {type(e).__name__}: {e}",
                    }
                f_out.write(json.dumps(result, ensure_ascii=False) + "\n")
                f_out.flush()
                if result.get("error") is None:
                    n_success += 1
                    total_configs += result.get("n_configs", 0)
                else:
                    n_error += 1

    elapsed = time.time() - start
    print(f"\nProcessados: {n_success + n_error}")
    print(f"  Sucesso:    {n_success}")
    print(f"  Erro:       {n_error}")
    print(f"  Configurações totais extraídas: {total_configs}")
    print(f"  Tempo: {elapsed:.1f}s ({(n_success+n_error)/max(elapsed,1):.2f} req/s)")
    return n_success + n_error

In [13]:
n_processed = run_extraction(df_ext, CKPT_PATH)

Total:   602
Sucessos prévios:   564
Restantes (inclui erros prévios):    38

Custo estimado dos restantes: ~$0.16
Tempo mínimo (limitado por 50 RPM): ~0.8 min



Extraindo: 100%|██████████| 38/38 [29:48<00:00, 47.06s/it]  


Processados: 38
  Sucesso:    38
  Erro:       0
  Configurações totais extraídas: 1054
  Tempo: 1788.5s (0.02 req/s)


## Achatar checkpoints em `configurations_raw.parquet`

O JSONL tem uma linha por paper (com uma lista de configurações dentro). Queremos uma linha por configuração:

- Para cada paper com sucesso, expandimos cada configuração em uma linha.
- Adicionamos `paper_id` e gerar `config_id` único (`{paper_id}_{idx}`).
- Salvamos como parquet.

In [14]:
def load_checkpoints_extraction(ckpt_path: Path) -> pd.DataFrame:
    rows = []
    if not ckpt_path.exists():
        return pd.DataFrame(rows)
    with open(ckpt_path, encoding="utf-8") as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except Exception:
                pass
    return pd.DataFrame(rows)


df_ckpt = load_checkpoints_extraction(CKPT_PATH)

df_ckpt["_is_success"] = df_ckpt["error"].isna() & df_ckpt["configurations"].notna()
df_ckpt["_order"] = range(len(df_ckpt))
df_ckpt = df_ckpt.sort_values(["paper_id", "_is_success", "_order"],
                               ascending=[True, True, True])
df_ckpt = df_ckpt.drop_duplicates("paper_id", keep="last").reset_index(drop=True)
df_ckpt = df_ckpt.drop(columns=["_is_success", "_order"])

print(f"Papers no checkpoint: {len(df_ckpt)}")
print(f"Com configurações: {df_ckpt['n_configs'].fillna(0).gt(0).sum()}")
print(f"Vazios (sem configurações utilizáveis): {df_ckpt['n_configs'].fillna(0).eq(0).sum() - df_ckpt['error'].notna().sum()}")
print(f"Com erro: {df_ckpt['error'].notna().sum()}")
print(f"Total de configurações extraídas: {int(df_ckpt['n_configs'].fillna(0).sum())}")

config_rows = []
for row in df_ckpt.itertuples():
    if not row.configurations:
        continue
    for idx, cfg in enumerate(row.configurations):
        flat = {"config_id": f"{row.paper_id}#{idx}", "paper_id": row.paper_id}
        flat.update(cfg)
        config_rows.append(flat)

df_configs = pd.DataFrame(config_rows)
df_configs.to_parquet(CONFIG_PATH, index=False)
print(f"\nSalvo em {CONFIG_PATH}")
print(f"Linhas (configurações): {len(df_configs)}")

Papers no checkpoint: 602
Com configurações: 512
Vazios (sem configurações utilizáveis): 90
Com erro: 0
Total de configurações extraídas: 17032

Salvo em c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\causal_project\data\processed\configurations_raw.parquet
Linhas (configurações): 17032


## Resumo

Quero ver:
- Distribuição de #configs por paper — esperamos uma cauda longa (alguns papers com 50+, outros com 2-3).
- Top datasets, modelos, estratégias mais frequentes (já em forma raw — vai dar pra perceber a normalização que vamos precisar).
- Distribuição de `extraction_confidence`.
- Quantos papers retornaram lista vazia + sample de `paper_notes`.

In [15]:
configs_per_paper = df_configs.groupby("paper_id").size()
print(f"Papers com >=1 config: {len(configs_per_paper)}")
print(f"Mediana de configs/paper: {configs_per_paper.median()}")
print(f"Média:                   {configs_per_paper.mean():.1f}")
print(f"Máx:                     {configs_per_paper.max()}")
print(f"\nDistribuição de configs/paper:")
print(configs_per_paper.describe([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

print(f"\nTop 10 datasets (raw — pré-normalização):")
print(df_configs["dataset_name_raw"].value_counts().head(10))

print(f"\nTop 10 modelos (raw):")
print(df_configs["model_name_raw"].value_counts().head(10))

print(f"\nTop 10 estratégias (raw):")
print(df_configs["balancing_strategy_raw"].value_counts().head(10))

print(f"\nTop 10 métricas (raw):")
print(df_configs["metric_name_raw"].value_counts().head(10))

print(f"\nDistribuição de extraction_confidence:")
print(df_configs["extraction_confidence"].describe())

print(f"\n% configurações marcadas como baseline:")
print(df_configs["is_baseline_raw"].value_counts(normalize=True))

Papers com >=1 config: 512
Mediana de configs/paper: 38.0
Média:                   33.3
Máx:                     44

Distribuição de configs/paper:
count    512.000000
mean      33.265625
std       10.861328
min        1.000000
25%       32.750000
50%       38.000000
75%       40.000000
90%       41.000000
95%       42.000000
99%       43.000000
max       44.000000
dtype: float64

Top 10 datasets (raw — pré-normalização):
dataset_name_raw
CIFAR100-LT     322
CIFAR-100-LT    281
ImageNet-LT     233
CIFAR-10-LT     197
Pima            152
CIFAR-10        130
Ionosphere      126
abalone9-18     126
Haberman        111
WI              105
Name: count, dtype: int64

Top 10 modelos (raw):
model_name_raw
SVM              1193
ResNet-32        1097
Random Forest    1012
RF                487
XGBoost           413
MLP               368
KNN               336
C4.5              318
DT                317
LR                260
Name: count, dtype: int64

Top 10 estratégias (raw):
balancing_strategy_r

In [16]:
empty_papers = df_ckpt[df_ckpt["n_configs"].fillna(0).eq(0) & df_ckpt["error"].isna()]
if len(empty_papers) > 0:
    print(f"\nPapers que retornaram lista vazia ({len(empty_papers)}):")
    print("Sample de paper_notes (até 5):")
    for row in empty_papers.head(5).itertuples():
        print(f"  - {row.paper_id}: {(row.paper_notes or '<sem nota>')[:200]}")

errors = df_ckpt[df_ckpt["error"].notna()]
if len(errors) > 0:
    print(f"\nTop categorias de erro ({len(errors)} total):")
    err_cat = errors["error"].astype(str).str.split(":").str[0]
    print(err_cat.value_counts().head(10))

total_in = int(df_ckpt["input_tokens"].fillna(0).sum())
total_out = int(df_ckpt["output_tokens"].fillna(0).sum())
cost = total_in / 1_000_000 * PRICE_INPUT_PER_M + total_out / 1_000_000 * PRICE_OUTPUT_PER_M
print(f"\nTokens totais — input: {total_in:>9,}  output: {total_out:>9,}")
print(f"Custo total estimado: ~${cost:.2f}")


Papers que retornaram lista vazia (90):
Sample de paper_notes (até 5):
  - arxiv:1707.03905: No explicit metric values are provided in the text. The paper describes experimental results using Dolan-More curves and a single example plot for one dataset, but no numerical metric values are repor
  - arxiv:1710.09515: <sem nota>
  - arxiv:1711.00837: The paper reports experimental results using 71 datasets (12 original UCI datasets plus undersampled and simulated variants), 3 classifiers (LR, KNN, GBM), 6 oversampling strategies (no oversampling, 
  - arxiv:1803.03877: <sem nota>
  - arxiv:1804.07155: No extractable experimental configurations found. The paper reports a sign test p-value matrix comparing 12 instance selection methods across 66 datasets, but does not provide per-dataset or per-metho

Tokens totais — input: 5,133,722  output: 3,741,602
Custo total estimado: ~$5.49
